# 10장. 글을 만드는 방법: 생성과 샘플링

학습된 모델로 이야기를 만들고, 만드는 방식을 조절해요. **GPU가 없어도 됩니다.** T4 GPU를 연결하면 글을 더 빨리 만들지만, 난수를 만드는 방식이 달라 책과 다른 글이 나와요. 책과 같은 결과를 보려면 CPU 런타임을 쓰세요.

드라이브에 9장의 `pretrain.pt`가 있으면 그것을, 없으면 저자의 체크포인트를 씁니다.


### 준비


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from torch.nn import functional as F
from modu_llm import Vocab, GPT, GPTConfig, load_checkpoint, get_file, mount_drive
from modu_llm import next_char_probs, generate, get_device

device = get_device()
vocab = Vocab.build()
folder = mount_drive()

model, _, step, history = load_checkpoint(get_file("pretrain.pt"), device)
model.eval()
print(f"{step}단계 학습한 모델")


## 10.1 다음 글자의 확률 들여다보기

모델은 글을 바로 쓰지 않아요. 매번 **다음 글자의 확률**을 계산하고, 그중 하나를 골라 이어 붙이는 일을 반복합니다.

**핵심 코드**: `next_char_probs(model, vocab, prompt)`


In [ ]:
for prompt in ["어느 날, 작은 ", "토끼는 기분이 좋아서 깡충깡충 뛰었어"]:
    print(f"[{prompt}]")
    for ch, p in next_char_probs(model, vocab, prompt, k=5):
        percent = p * 100
        print(f"   [{ch}] {percent:.1f}%")


이 확률표를 보고 글자를 하나씩 뽑아 이어 붙이는 생성 함수를 직접 만들어 봅니다.


In [ ]:
def my_generate(prompt, max_new_tokens=50):
    ids = vocab.encode(prompt)
    new_ids = []
    for i in range(max_new_tokens):
        context = torch.tensor([ids[-model.cfg.block_size:]], device=device)   # 마지막 384글자만
        with torch.no_grad():
            logits, _ = model(context)
        last = logits[0, -1]                       # 마지막 자리의 점수
        probs = F.softmax(last, dim=-1)            # 점수를 확률로
        next_id = torch.multinomial(probs, 1).item()   # 확률대로 하나 뽑기
        ids.append(next_id)
        new_ids.append(next_id)
    return vocab.decode(new_ids)

torch.manual_seed(0)
print("어느 날, " + my_generate("어느 날, "))


패키지의 `generate`는 이 함수에 세 가지를 더한 것이에요. 더한 것을 모두 끄면 결과가 같은지 확인해 봅시다.


In [ ]:
torch.manual_seed(0)
mine = my_generate("어느 날, ")
torch.manual_seed(0)
pkg = generate(model, vocab, "어느 날, ", max_new_tokens=50, temperature=1.0, stop_at_end=False)
print("두 함수의 결과가 같은가요?", mine == pkg)


## 10.2 온도: 대담하게, 조심스럽게

고르기 전에 점수를 **온도(temperature)**로 나눠요. 온도가 낮으면 확률 높은 글자에 더 몰리고(조심스럽게), 높으면 확률이 고르게 퍼집니다(대담하게).
같은 후보의 확률이 온도에 따라 어떻게 바뀌는지 먼저 보고, 실제 이야기도 비교해요.

**핵심 코드**: `last / temperature`


In [ ]:
prompt = "어느 날, 작은 "
idx = torch.tensor([vocab.encode(prompt)], device=device)
with torch.no_grad():
    logits, _ = model(idx)
last = logits[0, -1].float()
top = torch.topk(last, 5).indices

for t in [0.3, 0.8, 1.5]:
    probs = F.softmax(last / t, dim=-1)
    line = f"온도 {t}:"
    for i in top.tolist():
        ch = vocab.chars[i]
        percent = probs[i].item() * 100
        line = line + f"  [{ch}] {percent:.0f}%"
    print(line)


In [ ]:
for t in [0.3, 0.8, 1.5]:
    torch.manual_seed(0)
    story = generate(model, vocab, "어느 날, ", max_new_tokens=150, temperature=t)
    print(f"[온도 {t}] 어느 날, {story}")
    print()


## 10.3 상위 k개에서만 고르기

온도를 높이면 다양해지지만, 확률이 아주 낮은 엉뚱한 글자가 뽑히기도 해요. **상위 k개 글자 안에서만** 고르게 하면 이를 막을 수 있습니다.

**핵심 코드**: `top_k=10`


10-5의 점수(`last`)로 상위 3개만 남기는 계산을 직접 해 봐요. k등의 점수보다 낮은 글자는 4.4의 마스크처럼 `-inf`로 바꿉니다.


In [ ]:
t, k = 1.5, 3
scaled = last / t                                        # ① 온도로 나누기
kth = torch.topk(scaled, k).values[-1]                   # ② k등 글자의 점수
cut = scaled.masked_fill(scaled < kth, float("-inf"))    # ③ k등보다 낮으면 -inf
before = F.softmax(scaled, dim=-1)
after = F.softmax(cut, dim=-1)                           # ④ 확률로
for i in top.tolist():
    ch = vocab.chars[i]
    print(f"[{ch}] {before[i].item() * 100:.0f}% → {after[i].item() * 100:.0f}%")
print("확률이 0보다 큰 글자 수:", (before > 0).sum().item(), "→", (after > 0).sum().item())


패키지의 `generate`에 `top_k`를 주면 글자를 하나 뽑을 때마다 이 계산을 해요.


In [ ]:
for k in [None, 10]:
    torch.manual_seed(0)
    story = generate(model, vocab, "어느 날, ", max_new_tokens=150, temperature=1.3, top_k=k)
    print(f"[온도 1.3, top_k={k}] {story}")
    print()


## 10.4 이야기 끝에서 멈추기

2장에서 이야기 사이에 넣은 `§`를 모델은 "이야기가 끝나는 곳"으로 배웠어요. 멈추지 않게 하면 `§` 뒤에 새 이야기를 이어서 씁니다.

**핵심 코드**: `stop_at_end=True`


In [ ]:
torch.manual_seed(1)
story = generate(model, vocab, "어느 날, ", max_new_tokens=500, stop_at_end=False)
print("[멈추지 않기]", story)
print()
torch.manual_seed(1)
story = generate(model, vocab, "어느 날, ", max_new_tokens=500)
print("[§에서 멈추기]", story)


## 10.5 학습 단계별로 달라진 이야기

9장에서 저장한 단계별 체크포인트로 같은 시작 문장, 같은 난수로 글을 만들어 비교해요. 학습이 진행되며 무엇부터 배우는지 보세요.

**핵심 코드**: `load_checkpoint(get_file(name), device)`


In [ ]:
names = ["ckpt_step250.pt", "ckpt_step1000.pt", "ckpt_step3000.pt", "pretrain.pt"]

torch.manual_seed(0)
untrained = GPT(GPTConfig()).to(device)
story = generate(untrained, vocab, "어느 날, ", max_new_tokens=80)
print("[0단계]", story.replace("\n", " "))

for name in names:
    m, _, s, _ = load_checkpoint(get_file(name), device)
    torch.manual_seed(0)
    story = generate(m, vocab, "어느 날, ", max_new_tokens=80)
    print(f"[{s}단계]", story.replace("\n", " "))


## 10.6 그럴듯한 문장과 사실에 맞는 문장

이 모델은 어린이 이야기만 읽었어요. 사실을 묻는 문장을 넣으면 어떻게 될까요?
모델은 **사실인지가 아니라 그럴듯한 다음 글자**를 고릅니다. 큰 LLM이 그럴듯한 오답을 내는 것(환각)도 같은 원리에서 나와요.


In [ ]:
for prompt in ["대한민국의 수도는 ", "물은 몇 도에서 끓을까요? "]:
    torch.manual_seed(0)
    story = generate(model, vocab, prompt, max_new_tokens=100)
    print(prompt + story)
    print()


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

시작 문장과 설정을 바꿔 가며 나만의 이야기를 만들어 보세요.


In [ ]:
prompt = "겨울날, 눈사람이 "
story = generate(model, vocab, prompt, max_new_tokens=300, temperature=0.8, top_k=20)
print(prompt + story)


## 실제 LLM에서는

- 실제 서비스는 상위 k개 대신 누적 확률로 후보를 자르는 방식(top-p), 같은 말 반복을 줄이는 장치 등 여러 방법을 섞어 써요.
- 큰 모델도 "다음 토큰 확률"로 글을 만든다는 점은 같습니다. 그래서 답이 그럴듯해도 사실 확인이 필요해요.
